<a href="https://colab.research.google.com/github/munnurumahesh03-coder/kaggle-predicting-loan-payback/blob/main/04_Random_Forest_(LGBM_RF)_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Cell 1: Imports and Constants ---

# --- Core Libraries ---
import pandas as pd
import numpy as np
import os
import gc

# --- Machine Learning ---
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# --- Hyperparameter Tuning ---
import optuna

# --- Configuration and Constants ---
# Set a seed for reproducibility
SEED = 42

# Define the target variable
TARGET = 'loan_paid_back'

# Define the number of splits for cross-validation
N_SPLITS = 5

# Define the number of trials for Optuna.
# Random Forest is simpler than boosting models, so 20 is a good starting point.
N_TRIALS_SIMPLE = 20

# --- Suppress Optuna's informational messages ---
# This keeps the output clean during the tuning process.
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Verification ---
print("--- Cell 1: Imports and Constants ---")
print("All libraries imported and constants defined successfully.")
print(f"LightGBM version: {lgb.__version__}")
print(f"Optuna version: {optuna.__version__}")


--- Cell 1: Imports and Constants ---
All libraries imported and constants defined successfully.
LightGBM version: 4.6.0
Optuna version: 4.5.0


In [ ]:
# --- Cell 2 (v2): Load Data and Correctly Classify Binary Features ---

print("--- Cell 2 (v2): Loading and Preparing Data with Corrected Feature Types ---")

# --- Define File Paths ---
TRAIN_PATH = '/kaggle/input/02-feature-engineering-ipynb/train_featured_v2.csv'
TEST_PATH = '/kaggle/input/02-feature-engineering-ipynb/test_featured_v2.csv'

# --- Load Datasets ---
try:
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
    print("Datasets loaded successfully.")
    print(f"Train data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")
except FileNotFoundError:
    print("❌ ERROR: Data files not found. Please ensure Notebook 02's output is added as input to this notebook.")
    raise

# --- Prepare Data for Modeling ---
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]
X_test = test_df

# --- Identify Feature Types (Your Improvement) ---
# 1. Initially, separate columns by data type
categorical_features_initial = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features_initial = X.select_dtypes(include=np.number).columns.tolist()

# 2. Find binary (0/1) columns within the numerical list
binary_features = [col for col in numerical_features_initial if X[col].nunique() == 2 and X[col].min() == 0 and X[col].max() == 1]

# 3. Create the final, correct lists
# Your plan: Move binary features from numerical to categorical
numerical_features = [col for col in numerical_features_initial if col not in binary_features]
categorical_features = categorical_features_initial + binary_features

print(f"\nFound {len(binary_features)} binary (0/1) features: {binary_features}")
print("These will be treated as CATEGORICAL features as you suggested.")

# 4. The 'id' column is an identifier, not a feature
if 'id' in numerical_features:
    numerical_features.remove('id')
if 'id' in X_test.columns:
    test_ids = X_test['id']
    X_test = X_test.drop(columns=['id'])
else:
    test_ids = pd.Series(range(len(X_test)), name='id')

# --- Verification ---
print("\nData preparation complete.")
print(f"Number of features: {len(X.columns)}")
print(f"   - True Numerical features: {len(numerical_features)}")
print(f"   - Categorical (including binary): {len(categorical_features)}")
print(f"Target variable '{TARGET}' isolated.")
print(f"Test IDs captured. Shape: {test_ids.shape}")

# --- Clean up memory ---
del train_df, test_df
gc.collect()

print("\n--- Cell 2 (v2) Complete ---")


--- Cell 2 (v2): Loading and Preparing Data with Corrected Feature Types ---
Datasets loaded successfully.
Train data shape: (593994, 24)
Test data shape: (254569, 24)

Found 5 binary (0/1) features: ['is_unemployed', 'is_student', 'is_retired', 'is_home_or_business_loan', 'is_medical_or_edu_loan']
These will be treated as CATEGORICAL features as you suggested.

Data preparation complete.
Number of features: 23
   - True Numerical features: 12
   - Categorical (including binary): 11
Target variable 'loan_paid_back' isolated.
Test IDs captured. Shape: (254569,)

--- Cell 2 (v2) Complete ---


# **Automated Hyperparameter Tuning For LGBM-RF**

In [ ]:
# --- Cell 3 (v2): The ModelTuner Class with Real-Time Progress ---

print("--- Cell 3 (v2): Defining the ModelTuner Class with Real-Time Progress ---")

# This is the callback function. It will be called after every trial.
def print_trial_callback(study, trial):
    print(f"Trial {trial.number} finished with value: {trial.value:.6f} and parameters: {trial.params}")
    print(f"    Best value so far: {study.best_value:.6f}")

class ModelTuner:
    # ... (the __init__ and objective methods are EXACTLY THE SAME as before) ...
    def __init__(self, X, y, numerical_features, categorical_features, n_splits, seed):
        self.X = X
        self.y = y
        self.n_splits = n_splits
        self.seed = seed
        self.preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), numerical_features),('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)], remainder='passthrough')
        self.model = lgb.LGBMClassifier(boosting_type='rf', bagging_freq=1, device='gpu', random_state=self.seed, n_jobs=-1, verbosity=-1)
        self.params_search_space = lambda trial: {'classifier__n_estimators': trial.suggest_int('n_estimators', 100, 1000),'classifier__max_depth': trial.suggest_int('max_depth', 5, 20),'classifier__num_leaves': trial.suggest_int('num_leaves', 20, 100),'classifier__bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.99),'classifier__feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.99),'classifier__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1)}

    def objective(self, trial):
        pipeline = Pipeline(steps=[('preprocessor', self.preprocessor),('classifier', self.model)])
        params = self.params_search_space(trial)
        pipeline.set_params(**params)
        cv_scores = []
        skf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.seed)
        for train_idx, val_idx in skf.split(self.X, self.y):
            X_train, X_val = self.X.iloc[train_idx], self.X.iloc[val_idx]
            y_train, y_val = self.y.iloc[train_idx], self.y.iloc[val_idx]
            pipeline.fit(X_train, y_train)
            preds = pipeline.predict_proba(X_val)[:, 1]
            cv_scores.append(roc_auc_score(y_val, preds))
        return np.mean(cv_scores)

    # --- THIS IS THE ONLY PART THAT CHANGES ---
    def tune(self, n_trials):
        """Runs the Optuna study with a callback for real-time progress."""
        study = optuna.create_study(direction='maximize')
        # We add the 'callbacks' argument here
        study.optimize(self.objective, n_trials=n_trials, callbacks=[print_trial_callback])
        return study.best_params, study.best_value

print("ModelTuner class defined successfully.")
print("\n--- Cell 3 (v2) Complete ---")


--- Cell 3 (v2): Defining the ModelTuner Class with Real-Time Progress ---
ModelTuner class defined successfully.

--- Cell 3 (v2) Complete ---


In [ ]:
# --- Cell 4: Part 1 - Run Hyperparameter Tuning (Safe Version) ---

print("--- Cell 4: Starting Hyperparameter Tuning for LGBM-RF ---")

# 1. Create an instance of our ModelTuner robot.
tuner = ModelTuner(
    X=X,
    y=y,
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    n_splits=N_SPLITS,
    seed=SEED
)
print("ModelTuner instance created successfully.")

# 2. Start the tuning process.
print(f"\nStarting Optuna study with {N_TRIALS_SIMPLE} trials...")
print("This is the long-running part. Please be patient.")

# The tune() method returns the best parameters and the best score found.
# We store them in temporary variables first.
temp_best_params, temp_best_value = tuner.tune(n_trials=N_TRIALS_SIMPLE)

# --- Tuning Complete ---
print("\n" + "="*50)
print("--- Tuning Complete! ---")
print(f"✅ Best Score found: {temp_best_value:.6f}")
print(f"✅ Best Parameters found: {temp_best_params}")
print("\nThe results are now stored in temporary memory. Proceed to Cell 5 to save them permanently.")
print("="*50)


--- Cell 4: Starting Hyperparameter Tuning for LGBM-RF ---
ModelTuner instance created successfully.

Starting Optuna study with 20 trials...
This is the long-running part. Please be patient.
Trial 0 finished with value: 0.906250 and parameters: {'n_estimators': 593, 'max_depth': 5, 'num_leaves': 51, 'bagging_fraction': 0.6233698510619707, 'feature_fraction': 0.9651147186580307, 'learning_rate': 0.04298650205083788}
    Best value so far: 0.906250
Trial 1 finished with value: 0.911249 and parameters: {'n_estimators': 974, 'max_depth': 20, 'num_leaves': 34, 'bagging_fraction': 0.5256287202085288, 'feature_fraction': 0.621441336447851, 'learning_rate': 0.029975425888422426}
    Best value so far: 0.911249
Trial 2 finished with value: 0.912368 and parameters: {'n_estimators': 855, 'max_depth': 18, 'num_leaves': 62, 'bagging_fraction': 0.9309510833904902, 'feature_fraction': 0.8672934401101731, 'learning_rate': 0.06868059166980457}
    Best value so far: 0.912368
Trial 3 finished with valu

In [ ]:
# --- Cell 5 (RESCUE VERSION): Manually Fix Parameter Names ---

print("--- Cell 5 (RESCUE): Manually Fixing Parameter Names ---")

# This is the dictionary with the wrong format from Cell 4
print(f"Original (bad) params: {temp_best_params}")

# --- THIS IS THE FIX ---
# We will manually add the 'classifier__' prefix to every key in the dictionary.
corrected_params = {f'classifier__{key}': value for key, value in temp_best_params.items()}
# --- END OF FIX ---

print(f"\nCorrected (good) params: {corrected_params}")

# Create the dictionaries to hold all results
if 'best_params_all' not in locals():
    best_params_all = {}
if 'oof_preds_all' not in locals():
    oof_preds_all = {}
if 'test_preds_all' not in locals():
    test_preds_all = {}

# Save the CORRECTED parameters into our main dictionary
best_params_all['LGBM-RF'] = corrected_params

print("\n✅ Corrected best parameters successfully saved.")
print("\n--- Cell 5 (RESCUE) Complete ---")


--- Cell 5 (RESCUE): Manually Fixing Parameter Names ---
Original (bad) params: {'n_estimators': 396, 'max_depth': 10, 'num_leaves': 100, 'bagging_fraction': 0.7743627252472347, 'feature_fraction': 0.6492671412905807, 'learning_rate': 0.013147897766235796}

Corrected (good) params: {'classifier__n_estimators': 396, 'classifier__max_depth': 10, 'classifier__num_leaves': 100, 'classifier__bagging_fraction': 0.7743627252472347, 'classifier__feature_fraction': 0.6492671412905807, 'classifier__learning_rate': 0.013147897766235796}

✅ Corrected best parameters successfully saved.

--- Cell 5 (RESCUE) Complete ---


In [ ]:
# --- Cell 6 (v2): Generate Final Predictions (Corrected) ---

print("--- Cell 6 (v2): Generating OOF and Test Predictions for LGBM-RF ---")

# 1. Get the best parameters we saved in Cell 5
model_name = 'LGBM-RF'
# --- THIS IS THE KEY ---
# We use the parameters DIRECTLY from the dictionary. They already have the 'classifier__' prefix.
params = best_params_all[model_name]

# 2. Create a fresh instance of the model and the pipeline
final_model = lgb.LGBMClassifier(
    boosting_type='rf',
    bagging_freq=1,
    device='gpu',
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)
final_pipeline = Pipeline(steps=[('preprocessor', tuner.preprocessor),
                                 ('classifier', final_model)])

# 3. Set the best parameters found during tuning
#    Now, 'params' has the correct format (e.g., 'classifier__n_estimators')
final_pipeline.set_params(**params)
print("Final pipeline created and best parameters have been set correctly.")

# 4. Prepare for K-Fold prediction generation
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

print("\nStarting prediction generation across 5 folds...")

# 5. Loop through the folds to generate predictions
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"   - Processing Fold {fold + 1}/{N_SPLITS}...")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    final_pipeline.fit(X_train, y_train)
    oof_preds[val_idx] = final_pipeline.predict_proba(X_val)[:, 1]
    test_preds += final_pipeline.predict_proba(X_test)[:, 1] / N_SPLITS

# 6. Save the generated predictions into our main dictionaries
oof_preds_all[model_name] = oof_preds
test_preds_all[model_name] = test_preds

# 7. Calculate and display the final OOF score
final_oof_score = roc_auc_score(y, oof_preds)
print("\nPrediction generation complete.")
print(f"✅ Final OOF ROC AUC for {model_name}: {final_oof_score:.6f}")

# Clean up memory
gc.collect()

print("\n--- Cell 6 (v2) Complete ---")


--- Cell 6 (v2): Generating OOF and Test Predictions for LGBM-RF ---
Final pipeline created and best parameters have been set correctly.

Starting prediction generation across 5 folds...
   - Processing Fold 1/5...
   - Processing Fold 2/5...
   - Processing Fold 3/5...
   - Processing Fold 4/5...
   - Processing Fold 5/5...

Prediction generation complete.
✅ Final OOF ROC AUC for LGBM-RF: 0.914021

--- Cell 6 (v2) Complete ---


In [ ]:
# --- Cell 7: Save Predictions and Create Submission ---

print("--- Cell 7: Saving All Predictions and Creating Submission File ---")

# --- Part 1: Save OOF and Test Predictions for Stacking ---

# Create dataframes from our prediction dictionaries
# These will be used in our final ensembling notebook
oof_df = pd.DataFrame(oof_preds_all)
test_df = pd.DataFrame(test_preds_all)

# Save them to CSV files
oof_df.to_csv('oof_preds_rf.csv', index=False)
test_df.to_csv('test_preds_rf.csv', index=False)

print("✅ OOF and Test predictions for LGBM-RF saved to 'oof_preds_rf.csv' and 'test_preds_rf.csv'.")
print("\nOOF Predictions Head:")
print(oof_df.head())


# --- Part 2: Create Standalone Submission File ---

# Create a submission dataframe
submission_df = pd.DataFrame({'id': test_ids})

# Add the predictions from our LGBM-RF model
# This is the 'test_preds' variable from the previous cell
submission_df['loan_paid_back'] = test_preds

# Save the submission file
submission_df.to_csv('submission_LGBM-RF.csv', index=False)

print("\n✅ Submission file 'submission_LGBM-RF.csv' created successfully.")
print("\nSubmission File Head:")
print(submission_df.head())


# --- Final Step: Clean up memory ---
gc.collect()

print("\n--- Notebook 04 Complete ---")

--- Cell 7: Saving All Predictions and Creating Submission File ---
✅ OOF and Test predictions for LGBM-RF saved to 'oof_preds_rf.csv' and 'test_preds_rf.csv'.

OOF Predictions Head:
    LGBM-RF
0  0.923672
1  0.695195
2  0.907977
3  0.854160
4  0.910430

✅ Submission file 'submission_LGBM-RF.csv' created successfully.

Submission File Head:
       id  loan_paid_back
0  593994        0.890770
1  593995        0.922840
2  593996        0.370860
3  593997        0.907491
4  593998        0.914107

--- Notebook 04 Complete ---
